# Datapunt 2 - Segmentatie met CBS Open Data (K-Means)

Dit notebook laadt **Kerncijfers Wijken en Buurten** van het CBS, kiest automatisch bruikbare variabelen zoals inkomen en bevolkingsdichtheid, past **K-Means clustering** toe en maakt twee visualisaties plus clusterprofielen.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [ ]:
# ==========================================
# 1. INSTELLINGEN & FUNCTIES
# ==========================================
TABLE_ID = "85984NED"
BASE_URL = f"https://opendata.cbs.nl/ODataApi/OData/{TABLE_ID}"
RANDOM_STATE = 42
MAX_CBS_ROWS_PER_QUERY = 9_500

def get_odata(url, timeout=60):
    """Downloadt alle pagina's van een CBS OData endpoint en combineert die tot één DataFrame."""
    rows = []
    next_url = url
    while next_url:
        response = requests.get(next_url, timeout=timeout)
        response.raise_for_status()
        payload = response.json()
        if "value" in payload:
            chunk = payload["value"]
            next_url = payload.get("@odata.nextLink")
        elif "d" in payload:
            d = payload["d"]
            chunk = d.get("results", [])
            next_url = d.get("__next")
        else:
            raise RuntimeError("Onbekend OData-responseformaat ontvangen van CBS.")
        rows.extend(chunk)
    return pd.DataFrame(rows)

def prepare_text_columns(df, columns):
    for col in columns:
        if col in df.columns:
            df[col] = df[col].fillna("").astype(str).str.strip()
    return df

def find_key(metadata, include_terms=None, exclude_terms=None, key_prefixes=None, type_contains=None, required=False, label="veld"):
    include_terms = include_terms or []
    exclude_terms = exclude_terms or []
    key_prefixes = key_prefixes or []
    type_contains = type_contains or []

    meta = metadata.copy()
    title = meta.get("Title", pd.Series("", index=meta.index)).fillna("").astype(str)
    description = meta.get("Description", pd.Series("", index=meta.index)).fillna("").astype(str)
    key_series = meta.get("Key", pd.Series("", index=meta.index)).fillna("").astype(str)
    type_series = meta.get("Type", pd.Series("", index=meta.index)).fillna("").astype(str)

    search_text = (title + " " + description).str.lower()
    mask = pd.Series(True, index=meta.index)

    for term in include_terms:
        mask &= search_text.str.contains(term.lower(), regex=False)
    for term in exclude_terms:
        mask &= ~search_text.str.contains(term.lower(), regex=False)

    if key_prefixes:
        prefix_mask = pd.Series(False, index=meta.index)
        for prefix in key_prefixes:
            prefix_mask |= key_series.str.startswith(prefix)
        mask &= prefix_mask

    if type_contains:
        type_mask = pd.Series(False, index=meta.index)
        for term in type_contains:
            type_mask |= type_series.str.contains(term, case=False, regex=False)
        mask &= type_mask

    matches = meta[mask].copy()
    if not matches.empty:
        matches["title_len"] = matches["Title"].fillna("").astype(str).str.len()
        return matches.sort_values(["title_len", "Key"]).iloc[0]["Key"]

    if required:
        raise KeyError(f"Kon geen {label} vinden. include_terms={include_terms}, exclude_terms={exclude_terms}, key_prefixes={key_prefixes}, type_contains={type_contains}")
    return None

def choose_feature_keys(metadata):
    feature_specs = {
        "inkomen": [(["gestandaardiseerd inkomen"], [], []), (["gemiddeld inkomen"], [], []), (["besteedbaar inkomen"], [], [])],
        "dichtheid": [(["bevolkingsdichtheid"], [], []), (["omgevingsadressendichtheid"], [], [])],
        "inwoners": [(["aantal inwoners"], [], []), (["inwoners"], ["percentage"], [])],
        "vermogen": [(["mediaan vermogen"], [], []), (["vermogen particuliere huishoudens"], [], [])],
        "bijstand": [(["bijstand"], ["jeugd"], []), (["uitkering", "bijstand"], [], [])],
        "woningen": [(["woningvoorraad"], [], []), (["woningen"], ["gemiddeld"], [])],
        "ouderen": [(["65 jaar of ouder"], [], []), (["65 jaar"], ["gemiddelde leeftijd"], [])],
    }
    chosen = {}
    for logical_name, pattern_sets in feature_specs.items():
        for inc, exc, pref in pattern_sets:
            key = find_key(metadata, include_terms=inc, exclude_terms=exc, key_prefixes=pref, required=False, label=logical_name)
            if key and key not in chosen.values():
                chosen[logical_name] = key
                break
    return chosen

def build_typed_dataset_url(select_keys, filters=None, orderby=None, top=None):
    params = ["$format=json", f"$select={','.join(select_keys)}"]
    if filters:
        params.append(f"$filter={filters}")
    if orderby:
        params.append(f"$orderby={orderby}")
    if top is not None:
        params.append(f"$top={top}")
    return BASE_URL + "/TypedDataSet?" + "&".join(params)


def combine_filters(*filters):
    valid_filters = [flt for flt in filters if flt]
    if not valid_filters:
        return None
    return " and ".join(f"({flt})" for flt in valid_filters)


def format_odata_value(value):
    if isinstance(value, str):
        escaped = value.replace("'", "''")
        return f"'{escaped}'"
    if isinstance(value, (np.integer, int)):
        return str(int(value))
    if isinstance(value, (np.floating, float)):
        return str(float(value))
    raise TypeError(f"Niet-ondersteunde OData-waarde: {value!r}")

def is_cbs_query_limit_error(exc):
    response = getattr(exc, "response", None)
    if response is None:
        return False
    return response.status_code == 500 and "returns less than 10000 records" in response.text


def detect_latest_dimension_value(dimension_key, probe_prefixes, region_code_key):
    for prefix in probe_prefixes:
        probe_filter = f"startswith({region_code_key},'{prefix}')"
        probe_url = build_typed_dataset_url([dimension_key], filters=probe_filter, orderby=f"{dimension_key} desc", top=1)
        try:
            probe_df = get_odata(probe_url)
        except requests.HTTPError:
            continue
        if not probe_df.empty and dimension_key in probe_df.columns:
            return probe_df.iloc[0][dimension_key]

    fallback_url = build_typed_dataset_url([dimension_key], orderby=f"{dimension_key} desc", top=1)
    fallback_df = get_odata(fallback_url)
    if fallback_df.empty or dimension_key not in fallback_df.columns:
        raise RuntimeError(f"Kon geen waarde ophalen voor dimensie {dimension_key!r}.")
    return fallback_df.iloc[0][dimension_key]


def fetch_region_dataset(select_keys, region_code_key, prefixes, max_rows=MAX_CBS_ROWS_PER_QUERY, branch_chars="0123456789", extra_filter=None):
    """Laadt CBS-regio's in batches en splitst prefixes automatisch verder bij de CBS 10.000-recordlimiet."""
    batches = []
    pending_prefixes = list(prefixes)

    while pending_prefixes:
        prefix = pending_prefixes.pop(0)
        prefix_filter = f"startswith({region_code_key},'{prefix}')"
        filter_clause = combine_filters(extra_filter, prefix_filter)
        batch_url = build_typed_dataset_url(select_keys, filter_clause)

        try:
            batch_df = get_odata(batch_url)
        except requests.HTTPError as exc:
            if is_cbs_query_limit_error(exc):
                refined_prefixes = [f"{prefix}{char}" for char in branch_chars]
                if not refined_prefixes or refined_prefixes == [prefix]:
                    raise RuntimeError(
                        f"CBS-query voor prefix {prefix!r} blijft boven de limiet; kan niet verder verfijnen."
                    ) from exc
                pending_prefixes = refined_prefixes + pending_prefixes
                print(f"  - prefix {prefix}: te groot, opgesplitst in {len(refined_prefixes)} subprefixen")
                continue
            raise

        if batch_df.empty:
            continue
        if len(batch_df) >= max_rows:
            print(f"  - prefix {prefix}: waarschuwing, {len(batch_df):,} records opgehaald")
        else:
            print(f"  - prefix {prefix}: {len(batch_df):,} records")
        batches.append(batch_df)

    if not batches:
        raise RuntimeError("Er zijn geen CBS-regio's opgehaald met de gekozen filters.")

    combined = pd.concat(batches, ignore_index=True)
    combined = combined.drop_duplicates(subset=[region_code_key]).reset_index(drop=True)
    return combined

def elbow_k(inertias, k_values):
    """Gecorrigeerde elbow-heuristiek met normalisatie"""
    k_min, k_max = min(k_values), max(k_values)
    i_min, i_max = min(inertias), max(inertias)

    k_norm = [(k - k_min) / (k_max - k_min) for k in k_values]
    i_norm = [(i - i_min) / (i_max - i_min) for i in inertias]

    x1, y1 = k_norm[0], i_norm[0]
    x2, y2 = k_norm[-1], i_norm[-1]
    distances = []

    for x0, y0 in zip(k_norm, i_norm):
        numerator = abs((y2 - y1) * x0 - (x2 - x1) * y0 + x2 * y1 - y2 * x1)
        denominator = math.sqrt((y2 - y1) ** 2 + (x2 - x1) ** 2)
        distances.append(numerator / denominator)

    return k_values[int(np.argmax(distances))]

def get_title(metadata, key):
    rows = metadata.loc[metadata["Key"] == key, "Title"]
    return rows.iloc[0] if not rows.empty else key


In [ ]:
# ==========================================
# 2. METADATA & FEATURE SELECTIE
# ==========================================
print("Ophalen van metadata...")
metadata = get_odata(BASE_URL + "/DataProperties?$format=json")
metadata = prepare_text_columns(metadata, ["Title", "Description", "Key", "Type"])
metadata = metadata[metadata["Key"].fillna("").astype(str).str.strip() != ""].copy()

region_code_key = find_key(metadata, include_terms=["codering"], required=True, label="regiocode")
region_name_key = (
    find_key(metadata, include_terms=["wijknaam"], required=False, label="wijknaam")
    or find_key(metadata, include_terms=["buurtnaam"], required=False, label="buurtnaam")
    or find_key(metadata, include_terms=["gemeentenaam"], required=False, label="gemeentenaam")
    or find_key(metadata, include_terms=["naam"], exclude_terms=["straatnaam"], required=False, label="regionaam")
    or region_code_key
)
region_type_key = (
    find_key(metadata, include_terms=["soort regio"], required=False, label="soort regio")
    or find_key(metadata, include_terms=["regio"], exclude_terms=["regioaanduiding"], required=False, label="soort regio")
)

period_key = (
    find_key(metadata, key_prefixes=["Perioden"], required=False, label="periode")
    or find_key(metadata, include_terms=["perioden"], required=False, label="periode")
    or find_key(metadata, include_terms=["verslagjaar"], required=False, label="periode")
    or find_key(metadata, include_terms=["jaar"], exclude_terms=["65 jaar", "45 tot 65 jaar", "jonger dan 25 jaar"], required=False, label="periode")
)

chosen_features = choose_feature_keys(metadata)
if "inkomen" not in chosen_features or "dichtheid" not in chosen_features:
    raise RuntimeError("Essentiële clustering-features (inkomen en dichtheid) zijn niet gevonden.")

latest_period_filter = None
if period_key:
    latest_period_value = detect_latest_dimension_value(period_key, ["WK00", "WK0", "WK"], region_code_key)
    latest_period_filter = f"{period_key} eq {format_odata_value(latest_period_value)}"
    print(f"Gebruik meest recente CBS-periode: {latest_period_value}")


In [ ]:
# ==========================================
# 3. DATA DOWNLOAD (in batches onder de CBS limiet)
# ==========================================
selected_keys = [region_code_key, region_name_key]
if region_type_key:
    selected_keys.append(region_type_key)
selected_keys += list(chosen_features.values())
selected_keys = list(dict.fromkeys(selected_keys))

region_prefixes = [f"WK{i:02d}" for i in range(100)]
print("\nBezig met downloaden van de CBS dataset in batches per wijk-prefix...")
cbs_df = fetch_region_dataset(selected_keys, region_code_key, region_prefixes, extra_filter=latest_period_filter)
print("Ruwe CBS dataset-vorm:", cbs_df.shape)


In [ ]:
# ==========================================
# 4. DATA SCHOONMAKEN & IMPUTEREN
# ==========================================
text_cols = [region_code_key, region_name_key]
if region_type_key:
    text_cols.append(region_type_key)
cbs_df = prepare_text_columns(cbs_df, text_cols)

missing_markers = {".": np.nan, "..": np.nan, "x": np.nan, "-": np.nan, " ": np.nan, "": np.nan}
for col in cbs_df.columns:
    if col not in text_cols:
        cbs_df[col] = cbs_df[col].replace(missing_markers)
        cbs_df[col] = pd.to_numeric(cbs_df[col], errors="coerce")

feature_cols_all = list(chosen_features.values())
required_feature_keys = [chosen_features["inkomen"], chosen_features["dichtheid"]]
missing_ratio = cbs_df[feature_cols_all].isna().mean().sort_values()

feature_cols = []
for col in feature_cols_all:
    if col in required_feature_keys or missing_ratio[col] <= 0.35:
        feature_cols.append(col)

imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(
    imputer.fit_transform(cbs_df[feature_cols]),
    columns=feature_cols,
    index=cbs_df.index,
)

model_df = cbs_df[[region_code_key, region_name_key] + ([region_type_key] if region_type_key else [])].copy()
for col in feature_cols:
    model_df[col] = X_imputed[col].values

print("\nOpgeschoonde CBS dataset-vorm:", model_df.shape)

In [ ]:
# ==========================================
# 5. K-MEANS CLUSTERING & ELBOW
# ==========================================
X = model_df[feature_cols].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k_values = list(range(2, 11))
inertias = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

best_k = elbow_k(inertias, k_values)
print("\nAutomatisch gekozen aantal clusters (elbow):", best_k)

In [ ]:
# ==========================================
# 6. VISUALISATIES
# ==========================================
kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
model_df["cluster"] = kmeans.fit_predict(X_scaled)

# PCA Plot
pca = PCA(n_components=2, random_state=RANDOM_STATE)
components = pca.fit_transform(X_scaled)
plot_df = model_df.copy()
plot_df["pca_1"] = components[:, 0]
plot_df["pca_2"] = components[:, 1]

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
for cluster_id in sorted(plot_df["cluster"].unique()):
    subset = plot_df[plot_df["cluster"] == cluster_id]
    plt.scatter(subset["pca_1"], subset["pca_2"], label=f"Cluster {cluster_id}", alpha=0.7)
plt.xlabel("PCA component 1")
plt.ylabel("PCA component 2")
plt.title("Cluster-visualisatie in 2D (PCA)")
plt.legend()

# Income vs Density Plot
income_col = chosen_features["inkomen"]
density_col = chosen_features["dichtheid"]

plt.subplot(1, 2, 2)
for cluster_id in sorted(model_df["cluster"].unique()):
    subset = model_df[model_df["cluster"] == cluster_id]
    plt.scatter(subset[income_col], subset[density_col], label=f"Cluster {cluster_id}", alpha=0.7)
plt.xlabel(get_title(metadata, income_col))
plt.ylabel(get_title(metadata, density_col))
plt.title("Clusters: Inkomen vs Dichtheid")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 7. CONCLUSIES
# ==========================================
profile_df = model_df.groupby("cluster")[feature_cols].mean().round(2)

def cluster_label(cluster_id, row, overall_means, income_col, density_col):
    income_value = row[income_col]
    density_value = row[density_col]
    if income_value >= overall_means[income_col] and density_value >= overall_means[density_col]:
        short_desc = "hoger inkomen en hogere dichtheid"
    elif income_value >= overall_means[income_col] and density_value < overall_means[density_col]:
        short_desc = "hoger inkomen en lagere dichtheid"
    elif income_value < overall_means[income_col] and density_value >= overall_means[density_col]:
        short_desc = "lager inkomen en hogere dichtheid"
    else:
        short_desc = "lager inkomen en lagere dichtheid"
    return f"Cluster {cluster_id}: {short_desc}."

overall_means = model_df[feature_cols].mean()

print("\nClusterprofielen:")
print("-" * 20)
for cluster_id, row in profile_df.iterrows():
    print(cluster_label(cluster_id, row, overall_means, income_col, density_col))
    example_regions = model_df[model_df["cluster"] == cluster_id][region_name_key].head(5).tolist()
    print("Voorbeelden:", ", ".join(example_regions))
    print()

print("Technische conclusie")
print("--------------------")
print(f"Er zijn {len(model_df):,} regio's gebruikt voor clustering.")
print(f"De elbow-methode koos k = {best_k} clusters.")
print(f"Het model gebruikt {len(feature_cols)} kenmerken.")
print("Ontbrekende waarden zijn met de mediaan geïmputeerd.")